# Generalized linear models

_Machine Learning_

**One framework that unifies linear and logistic regression.**

Linear and logistic regression feel different. But they share one recipe.

     Both compute a linear score, then connect it to a probability distribution.

---

This notebook builds up a generalized linear model one step at a time. Run each cell, read the note above it, and see how picking the right distribution and link makes a model bend to its data. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

A GLM computes a linear score and then connects it to a probability distribution through a **link function**. We build it in three steps: (1) generate count data with a known log link, (2) fit a Poisson GLM that matches that process, (3) fit plain linear regression to see why the distribution choice matters.

### Step 1 — Generate Poisson count data

We make a dataset whose target is a **count** that grows exponentially with `x`: the true mean is `exp(0.5*x)`, and each `y` is a Poisson draw around that mean. This `exp` relationship is exactly the **log link** a Poisson GLM assumes, so we know the right model in advance.

In [ ]:
import numpy as np

# count data: y ~ Poisson(mean = exp(0.5*x)), a log link
rng = np.random.default_rng(0)
X = rng.uniform(0, 3, size=(300, 1))
mean = np.exp(0.5 * X[:, 0])   # the true (log-link) mean for each x
y = rng.poisson(mean)          # noisy integer counts around that mean

### Step 2 — Fit a Poisson GLM

A `PoissonRegressor` assumes the target is Poisson-distributed with a log link — the same recipe that generated the data. We set `alpha=0.0` to turn off regularisation so the fit is driven purely by the data. The recovered coefficient should land near the true `0.5`.

In [ ]:
from sklearn.linear_model import PoissonRegressor

# Poisson GLM matches the data-generating process
glm = PoissonRegressor(alpha=0.0).fit(X, y)

print("Poisson coef:", round(glm.coef_[0], 3), " intercept:", round(glm.intercept_, 3))
print("Poisson deviance score:", round(glm.score(X, y), 4))

### Step 3 — Compare against plain linear regression

Plain linear regression assumes a Gaussian target and a straight-line relationship, ignoring the exponential count structure. Its `R^2` here is worse, showing that matching the distribution and link to the data — the whole point of GLMs — actually buys a better fit.

In [ ]:
from sklearn.linear_model import LinearRegression

# plain linear regression ignores the count structure -> worse fit
lin = LinearRegression().fit(X, y)
print("linear R^2:", round(lin.score(X, y), 4))

## Visualize the data & results

_Diabetes progression is a non-negative count-like target. Does a Poisson GLM (log link) bend with it better than a straight line?_

### Step 1 — Load and prepare the diabetes data

We use the real diabetes dataset and treat disease progression as a non-negative count. We pull out the BMI feature and **shift it to start at 0** so the values are easy to plot and interpret on the x-axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

# real diabetes data; progression treated as a non-negative count
db = load_diabetes()
x = db.data[:, 2:3]                # column 2 is BMI, kept as a 2D column
x = x - x.min()                    # shift BMI to start at 0
y = db.target.astype(int)          # progression as integer counts

### Step 2 — Fit both models

We fit a Poisson GLM (log link) and a plain linear regression to the same BMI feature. We bump `max_iter` up so the GLM's iterative solver fully converges on this real, noisier data.

In [ ]:
from sklearn.linear_model import PoissonRegressor, LinearRegression

glm = PoissonRegressor(alpha=0.0, max_iter=2000).fit(x, y)   # log link
lin = LinearRegression().fit(x, y)

### Step 3 — Plot the two fits over the data

We build a smooth grid of BMI values, predict progression from each model along it, and overlay both curves on the scatter of patients. The Poisson curve bends upward (its log link), while the linear fit is forced to stay straight — letting you see which one tracks the data better.

In [ ]:
# A smooth grid of BMI values to draw each model's prediction curve.
xs = np.linspace(x.min(), x.max(), 30).reshape(-1, 1)

plt.scatter(x[:, 0], y, c="#4ea1ff", edgecolor="k", label="patients")
plt.plot(xs[:, 0], glm.predict(xs), color="#7ee787", label="Poisson (log link)")
plt.plot(xs[:, 0], lin.predict(xs), color="#ff7b72", label="linear")
plt.xlabel("BMI (shifted)")
plt.ylabel("disease progression")
plt.title("Poisson GLM vs linear fit")
plt.legend()
plt.show()